# Route Testing

Quick smoke tests for the backend API. Make sure the server is running on `localhost:8000` before executing.

In [1]:
import httpx

BASE_URL = "http://localhost:8000"

In [ ]:
# Health Check

resp = httpx.get(f"{BASE_URL}/health")
print(f"Status: {resp.status_code}")
resp.json()

Status: 200


{'status': 'ok'}

In [ ]:
# POST /trace_contributions

resp = httpx.post(
    f"{BASE_URL}/trace_contributions",
    json={"repo_url": "https://github.com/rayedchow/codeu"},
)
print(f"Status: {resp.status_code}")
data = resp.json()

print(f"\nRepo: {data['repo']}")
print(f"Top {len(data['top_contributors'])} contributors:\n")
for i, c in enumerate(data["top_contributors"], 1):
    print(f"  {i:>2}. {c['username']:<30} {c['contributions']:>6} commits")

Status: 200

Repo: rayedchow/codeu
Top 3 contributors:

   1. rayedchow                          25 commits
   2. aaronbiscotti                      18 commits
   3. Mr-EZO949                           1 commits


In [27]:
resp = httpx.post(
    f"{BASE_URL}/trace_graph",
    json={
        "repo_url": "https://github.com/pallets/flask",
        "max_depth": 3,
        "max_children": 5,
    },
    timeout=180.0,
)
print(f"Status: {resp.status_code}")
data = resp.json()

print(f"\nRepo: {data['repo']}")
print(f"Config: depth={data['config']['max_depth']}, children={data['config']['max_children']}")
print(f"Graph: {len(data['graph']['nodes'])} nodes, {len(data['graph']['edges'])} edges\n")

# Print nodes by type
for ntype in ("REPO", "BODY_OF_WORK", "CONTRIBUTOR"):
    nodes = [n for n in data["graph"]["nodes"] if n["type"] == ntype]
    print(f"  {ntype} ({len(nodes)}):")
    for n in nodes[:10]:
        print(f"    - {n['label']}")
    if len(nodes) > 10:
        print(f"    ... and {len(nodes) - 10} more")
    print()

Status: 200

Repo: pallets/flask
Config: depth=3, children=5
Graph: 42 nodes, 43 edges

  REPO (1):
    - pallets/flask

  BODY_OF_WORK (11):
    - Direct Code
    - pallets/werkzeug
    - Direct Code
    - pallets/click
    - Direct Code
    - pallets/jinja
    - Direct Code
    - pallets/markupsafe
    - Direct Code
    - tartley/colorama
    ... and 1 more

  CONTRIBUTOR (30):
    - davidism
    - mitsuhiko
    - untitaker
    - rduplain
    - greyli
    - davidism
    - mitsuhiko
    - lkarthee
    - aenglander
    - ThomasWaldmann
    ... and 20 more



### Visualize edge weights

In [28]:
# Print the graph as a tree
def print_tree(data):
    graph = data["graph"]
    node_map = {n["id"]: n for n in graph["nodes"]}
    children = {}
    for e in graph["edges"]:
        children.setdefault(e["source"], []).append(e)

    def walk(node_id, indent=0):
        node = node_map.get(node_id, {})
        label = node.get("label", node_id)
        ntype = node.get("type", "?")
        prefix = "  " * indent
        print(f"{prefix}[{ntype}] {label}")
        for edge in sorted(children.get(node_id, []), key=lambda e: -e["weight"]):
            pct = f"{edge['weight']*100:.1f}%"
            target = node_map.get(edge["target"], {})
            target_label = target.get("label", edge["target"])
            target_type = target.get("type", "?")
            if target_type == "CONTRIBUTOR":
                print(f"{prefix}  └─ {pct:>6s}  @{target_label}")
            else:
                print(f"{prefix}  ├─ {pct:>6s}  {target_label}")
                walk(edge["target"], indent + 2)

    root = [n for n in graph["nodes"] if n["type"] == "REPO"]
    if root:
        walk(root[0]["id"])

print_tree(data)

[REPO] pallets/flask
  ├─  60.0%  Direct Code
    [BODY_OF_WORK] Direct Code
      └─  51.7%  @davidism
      └─  34.0%  @mitsuhiko
      └─   7.8%  @untitaker
      └─   3.5%  @rduplain
      └─   3.0%  @greyli
  ├─  18.1%  pallets/werkzeug
    [BODY_OF_WORK] pallets/werkzeug
      ├─  60.0%  Direct Code
        [BODY_OF_WORK] Direct Code
          └─  40.1%  @mitsuhiko
          └─  37.1%  @davidism
          └─  14.7%  @untitaker
          └─   5.1%  @DasIch
          └─   3.0%  @pgjones
      ├─  40.0%  pallets/markupsafe
        [BODY_OF_WORK] pallets/markupsafe
          ├─ 100.0%  Direct Code
            [BODY_OF_WORK] Direct Code
              └─  86.4%  @davidism
              └─  11.7%  @mitsuhiko
              └─   1.0%  @edgarrmondragon
              └─   0.5%  @methane
              └─   0.3%  @ThiefMaster
  ├─   8.2%  pallets/click
    [BODY_OF_WORK] pallets/click
      ├─  60.0%  Direct Code
        [BODY_OF_WORK] Direct Code
          └─  46.7%  @davidism
          └─  

### Verify edge weight invariant (outgoing edges sum to 1.0 per node)

In [29]:
from collections import defaultdict

outgoing = defaultdict(float)
for e in data["graph"]["edges"]:
    outgoing[e["source"]] += e["weight"]

print("Outgoing weight sums per node:")
all_ok = True
for node_id, total in sorted(outgoing.items()):
    status = "OK" if abs(total - 1.0) < 0.02 else "WARN"
    if status == "WARN":
        all_ok = False
    label = next((n["label"] for n in data["graph"]["nodes"] if n["id"] == node_id), node_id)
    print(f"  {status:4s}  {total:.4f}  {label}")

print(f"\nAll nodes within tolerance: {all_ok}")

Outgoing weight sums per node:
  OK    0.9999  Direct Code
  OK    1.0000  Direct Code
  OK    1.0000  Direct Code
  OK    1.0000  Direct Code
  OK    1.0000  Direct Code
  OK    1.0000  Direct Code
  OK    1.0000  pallets/click
  OK    1.0000  pallets/jinja
  OK    1.0000  pallets/markupsafe
  OK    1.0000  pallets/werkzeug
  OK    1.0000  tartley/colorama
  OK    1.0000  pallets/flask

All nodes within tolerance: True
